# Process Documents

In [2]:
import os

import pandas as pd
from docling.document_converter import DocumentConverter

from agent_k.config.logger import logger

df_inferlink_gt = pd.read_csv("data/processed/inferlink_ground_truth_filtered.csv")
record_ids = df_inferlink_gt["cdr_record_id"].tolist()

# Search each record ID in the 43-101 directory "data/raw/all_sources/43-101"
pdf_paths = []
for record_id in record_ids:
    for file in os.listdir("data/raw/all_sources/43-101"):
        if record_id in file:
            pdf_paths.append(os.path.join("data/raw/all_sources/43-101", file))

converter = DocumentConverter()

for i, pdf_path in enumerate(pdf_paths):
    logger.info(f"{i}/{len(pdf_paths)}: Processing {pdf_path}")

    # Check if the markdown file already exists
    record_id = pdf_path.split("/")[-1].split(".")[0]
    if os.path.exists(os.path.join("data/processed/43-101", f"{record_id}.md")):
        logger.info(f"Skipping {record_id} because it already exists")
        continue

    result = converter.convert(pdf_path)
    # Save the markdown to a file in processed/43-101
    with open(os.path.join("data/processed/43-101", f"{record_id}.md"), "w") as f:
        f.write(result.document.export_to_markdown())

2025-03-14 17:48:41.655 | INFO     | __main__:<module>:20 - 0/31: Processing data/raw/all_sources/43-101/0200a1c6d2cfafeb485d815d95966961d4c119e8662b8babec74e05b59ba4759d2.pdf
2025-03-14 17:49:57.652 | INFO     | __main__:<module>:20 - 1/31: Processing data/raw/all_sources/43-101/0204fc707f5b1944308624520cd422c4f1cb478046f66487b06a27e8fbd8f0fd86.pdf
2025-03-14 17:51:47.673 | INFO     | __main__:<module>:20 - 2/31: Processing data/raw/all_sources/43-101/02163e7674b5c838a0efd7d2e80116e36ba01cbdb1d892921ec50e12dc041de809.pdf
2025-03-14 17:56:20.657 | INFO     | __main__:<module>:20 - 3/31: Processing data/raw/all_sources/43-101/02195e2ef16f106876aff4c08b7a53a04edc8630e7ee0a23f16b61a8b066892e8e.pdf
2025-03-14 17:59:07.669 | INFO     | __main__:<module>:20 - 4/31: Processing data/raw/all_sources/43-101/021ee3d3ec6928901d9427796372edaada0d332f15d03040570ca758e1b7f06445.pdf
2025-03-14 18:04:24.691 | INFO     | __main__:<module>:20 - 5/31: Processing data/raw/all_sources/43-101/023a9e77a67e368

## Refine Headers
1. Removing false positive headers in markdown splitters (e.g. "## Notes:")
2. Correct header levels

In [5]:
import os
import re
from pathlib import Path


def extract_headers_from_markdown(markdown_text):
    """
    Extract all headers from a markdown text.

    Args:
        markdown_text (str): The markdown text to parse

    Returns:
        list: A list of tuples containing (header_level, header_text)
    """
    # Regular expression to match markdown headers (# Header)
    header_pattern = re.compile(r"^(#{1,6})\s+(.+?)(?:\s+#+)?$", re.MULTILINE)

    # Find all headers in the text
    headers = []
    for match in header_pattern.finditer(markdown_text):
        level = len(match.group(1))  # Number of # characters
        text = match.group(2).strip()
        headers.append((level, text))

    return headers


# Example usage
markdown_dir = Path("data/processed/43-101")
sample_files = list(markdown_dir.glob("*.md"))[
    4:7
]  # Get first 3 files for demonstration

for file_path in sample_files:
    print(f"\nHeaders in {file_path.name}:")

    with open(file_path, "r", encoding="utf-8") as f:
        markdown_content = f.read()

    headers = extract_headers_from_markdown(markdown_content)

    for level, text in headers:
        print(f"{'  ' * (level - 1)}{'#' * level} {text}")


Headers in 02d40055ac23d9034c815a941763045780b709289537ed42a56b7390db074d11cd.md:
  ## Technical Report for the McWatters Nickel Deposit, Ontario, Canada
  ## Liberty Mines Inc
  ## Technical Report for the McWatters Nickel Deposit in Ontario, Canada
  ## Liberty Mines Inc
  ## 16 May 2008
  ## Executive Summary
  ## Introduction
  ## Work Program
  ## Property Description
  ## Geology and Mineralization
  ## Mineral processing and metallurgy
  ## Mineral Resources
  ## Mineral Reserves
  ## Mining Operations
  ## Environmental
  ## McWatters Life of Mine Plan
  ## Summary and Recommendations
  ## Table of Contents
  ## List of Tables
  ## 1 Introduction
  ## 1.1 Background of the project
  ## 1.2 Qualification of SRK
  ## 1.3 Project team
  ## 1.4 Basis of the Technical Report
  ## 1.5 Site visit
  ## 2 Reliance on other Experts
  ## 3 Property Description and Location
  ## 3.1 Land Tenure
  ## 3.2 Underlying Agreements
  ## 4 Accessibility, Climate, Local Resources, Infrastructure a

In [14]:
def correct_header_levels(headers):
    """
    Correct header levels in markdown headers based on specified heuristics:
    1. By default keep header level as level 2
    2. Numbered headers get levels based on their numbering depth relative to the highest level:
       - If highest level is "1", then "1" -> level 2, "1.1" -> level 3, etc.
       - If highest level is "1.0", then "1.0" -> level 2, "1.0.1" -> level 3, etc.
    3. Unnumbered headers between numbered headers get level one below the enclosing numbered headers
    4. Replace "## Notes:", "## Notes", or "## Note" with "Notes:"

    Args:
        headers: List of tuples (header_level, header_text)

    Returns:
        List of tuples with corrected (header_level, header_text)
    """
    if not headers:
        return []

    import re

    # First pass: Determine the highest level numbered header pattern
    highest_level_parts = None
    numbered_headers_info = []

    for i, (_, text) in enumerate(headers):
        number_match = re.match(r"^(\d+(\.\d+)*)(\s+|\.|\s*$)", text)
        if number_match:
            number = number_match.group(1)
            parts = number.split(".")

            # Store information about this numbered header
            numbered_headers_info.append((i, number, parts))

            # Update highest_level_parts if this is a higher level
            if highest_level_parts is None or len(parts) < len(highest_level_parts):
                highest_level_parts = parts

    # If no numbered headers found, highest level is None
    base_level_length = len(highest_level_parts) if highest_level_parts else 0

    # Second pass: Process all headers with the determined base level
    corrected = []
    current_section_level = None

    for i in range(len(headers)):
        original_level, text = headers[i]

        # Check if this is a numbered header
        number_match = re.match(r"^(\d+(\.\d+)*)(\s+|\.|\s*$)", text)

        if number_match:
            # This is a numbered header
            number = number_match.group(1)
            parts = number.split(".")

            # Calculate level relative to the highest level
            # The highest level becomes level 2
            level_difference = len(parts) - base_level_length
            level = 2 + level_difference

            current_section_level = level
            corrected.append((level, text))
        else:
            # This is an unnumbered header

            # Check if there's a numbered header after this one
            has_following_numbered = False
            next_numbered_level = None

            for j in range(i + 1, len(headers)):
                next_text = headers[j][1]
                next_match = re.match(r"^(\d+(\.\d+)*)(\s+|\.|\s*$)", next_text)
                if next_match:
                    has_following_numbered = True
                    next_number = next_match.group(1)
                    next_parts = next_number.split(".")

                    # Calculate level using the same relative approach
                    level_difference = len(next_parts) - base_level_length
                    next_numbered_level = 2 + level_difference
                    break

            # Between numbered headers
            if current_section_level is not None and has_following_numbered:
                # It's between two numbered headers
                # Assign level one below the minimum of the enclosing levels
                enclosing_level = min(current_section_level, next_numbered_level)
                corrected.append((enclosing_level + 1, text))
            else:
                # Default to level 2
                corrected.append((2, text))

        # Replace "## Notes:", "## Notes", or "## Note" with "Notes:"
        if text.lower() in ["## notes:", "## notes", "## note"]:
            corrected.append((2, "Notes:"))

    return corrected


if __name__ == "__main__":
    markdown_dir = Path("data/processed/43-101")
    sample_files = list(markdown_dir.glob("*.md"))[4]  # Get first file
    with open(sample_files, "r", encoding="utf-8") as f:
        markdown_content = f.read()
    headers = extract_headers_from_markdown(markdown_content)
    corrected_headers = correct_header_levels(headers)

    print("\nCorrected headers:")
    for level, text in corrected_headers:
        print(f"{'  ' * (level - 1)}{'#' * level} {text}")


Corrected headers:
  ## Technical Report for the McWatters Nickel Deposit, Ontario, Canada
  ## Liberty Mines Inc
  ## Technical Report for the McWatters Nickel Deposit in Ontario, Canada
  ## Liberty Mines Inc
  ## 16 May 2008
    ### Executive Summary
    ### Introduction
    ### Work Program
    ### Property Description
    ### Geology and Mineralization
    ### Mineral processing and metallurgy
    ### Mineral Resources
    ### Mineral Reserves
    ### Mining Operations
    ### Environmental
    ### McWatters Life of Mine Plan
    ### Summary and Recommendations
    ### Table of Contents
    ### List of Tables
  ## 1 Introduction
    ### 1.1 Background of the project
    ### 1.2 Qualification of SRK
    ### 1.3 Project team
    ### 1.4 Basis of the Technical Report
    ### 1.5 Site visit
  ## 2 Reliance on other Experts
  ## 3 Property Description and Location
    ### 3.1 Land Tenure
    ### 3.2 Underlying Agreements
  ## 4 Accessibility, Climate, Local Resources, Infrastructure a

In [18]:
# Write a function to correct and replace the headers in all markdown files in the data/processed/43-101 directory and write the corrected markdown to a new folder in the data/processed/43-101-header-corrected directory
import os

os.makedirs("data/processed/43-101-header-corrected", exist_ok=True)


def correct_headers_in_directory(directory_path):
    for file_path in Path(directory_path).glob("*.md"):
        with open(file_path, "r", encoding="utf-8") as f:
            markdown_content = f.read()
        headers = extract_headers_from_markdown(markdown_content)
        corrected_headers = correct_header_levels(headers)
        for old_header, new_header in zip(headers, corrected_headers, strict=False):
            markdown_content = markdown_content.replace(
                f"{'#' * old_header[0]} {old_header[1]}",
                f"{'#' * new_header[0]} {new_header[1]}",
            )
        with open(
            os.path.join("data/processed/43-101-header-corrected", file_path.name),
            "w",
            encoding="utf-8",
        ) as f:
            f.write(markdown_content)


correct_headers_in_directory("data/processed/43-101")